# Pitch benchmark

Run All refreshes Attune with the production sweep defaults (live and offline: volume 0.06, unvoiced threshold 0.9875). Compatible Attune predictions/checkpoints are reused. Every competitor remains cached; missing competitor rows stop execution. Bach10/URMP tuning source pieces are excluded; all Coco evaluation stems are retained. Original reports remain immutable. Restart the kernel before Run All.


### Fair competitor configuration

Every external estimate is now given the same reference/score-derived admissible F0 range. APIs that support range-aware candidate selection (for example Attune, Praat, and Basic Pitch) use it internally; fixed-output APIs are also range-gated after inference. This distinction matters for SwiftF0: its [official API](https://github.com/lars76/swift-f0/) says the network always detects over 46.875–2093.75 Hz and `fmin`/`fmax` only decide which outputs count as voiced. The shared final gate therefore supplies the same information without claiming that it changes SwiftF0's CNN inference. Praat remains raw AC because Praat documents it as the periodicity method; its filtered recommendation is aimed at speech intonation and vocal-fold analysis.

SPICE uses one fixed confidence threshold of **0.35** across every dataset and the streaming adapter. Predictions below it remain unvoiced for scoring. The threshold was chosen after the exploratory confidence sweep; these preliminary results are not a held-out estimate of threshold selection performance.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "app.py").is_file()
    )
)
sys.path.insert(0, str(REPO_ROOT))
import importlib

for module_name in (
    "algorithms.PitchSmoother",
    "benchmarks.modules.pitch.PitchCache",
    "benchmarks.modules.pitch.competitors.Attune",
    "benchmarks.modules.pitch.competitors.Spice",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.datasets.Bach10",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchNotebook",
):
    importlib.reload(importlib.import_module(module_name))
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook


In [3]:
PRIMARY_METHODS = ("pyin", "attune", "praat", "spice")
RUN_FULL_RUNS = False
CONFIG = PitchNotebook.NotebookConfig(use_cache=True, force_rerun=False, attune_only_refresh=True)
from benchmarks.notebooks.scripts.evaluation_holdout import pitch_notebook, Holdout
BENCH = pitch_notebook(CONFIG)
CACHE_METHODS = tuple(BENCH.methods)
BENCH.show_configuration()
RUN_PAPER_EVALUATION = True
OFFLINE_METHODS = tuple(BENCH.methods)
ONLINE_METHODS = ("attune", "pyin_framewise", "praat", "swiftf0", "spice", "mcleod", "crepe_tiny")
PAPER_ONLINE_METHODS = ONLINE_METHODS
PAIRED_RESAMPLES = 9999


,setting,value
0,workers,8
1,streaming workers,8
2,preliminary tracks per instrument,2
3,full CocoChorales stems per stratum,all
4,random seed,0
5,methods,"pyin, attune, basic_pitch, praat, swiftf0, cre..."
6,estimate caches,reused
7,degradation SNRs,"clean, 20dB, 15dB, 10dB, 5dB"
8,run cache,/Users/sarah/Desktop/shasha/Attune/benchmarks/...


## Method availability

Third-party trackers are optional dependencies, and a missing one is skipped rather than fatal. Run this first so a long cell later does not quietly drop a method: anything marked `MISSING` carries the install hint that would make it available. `torchcrepe` and `penn` are excluded by default because they stack a second heavy PyTorch runtime; set `include_slow_methods=True` on the config to include them, ideally in their own invocation.

In [4]:
availability = BENCH.show_availability()


,method,status,description,detail
0,pyin,OK,Local librosa-compatible pYIN at Attune's 44.1...,
1,attune,OK,Attune pYIN+ promoted production pipeline,
2,basic_pitch,OK,"Basic Pitch contour head, projected to one sub...",
3,praat,OK,Praat autocorrelation (praat-parselmouth),
4,swiftf0,OK,SwiftF0 CNN (swift-f0),
5,crepe,OK,"CREPE CNN, TensorFlow reference implementation",
6,spice,OK,SPICE self-supervised pitch estimator (TF Hub),
7,rmvpe,OK,RMVPE deep U-Net (vendored),
8,mcleod,OK,"Tartini 1.2 native MPM (stateful, official sou...",


## Preliminary sampling check

Every preliminary suite uses the same sampling contract: `CONFIG.preliminary_tracks_per_instrument` tracks for every instrument label, selected independently within each label using `CONFIG.seed`. With the defaults this yields 8 original Bach10 isolated recordings (4 instruments × 2), 26 URMP stems (13 × 2), and 26 CocoChorales stems (13 × 2). The check below also confirms that the same seed repeats exactly while the next seed changes track identities without changing those quotas.

In [5]:
# Frozen source-piece exclusions apply to all returned results.
import pandas as pd
display(pd.DataFrame([{'dataset': d, 'excluded_source_pieces': len(p)}
                      for d, p in BENCH.holdout.pieces.items()]))
print('Filtered report root:', BENCH.holdout.output)


,dataset,excluded_source_pieces
0,urmp,11
1,bach10-original,2


Filtered report root: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/evaluation_holdout/ae55a96844350db9


## Preliminary comparison — original Bach10 recordings

Two seed-selected recordings from each of Bach10's four instruments (violin, clarinet, saxophone, bassoon; 8 total). Only isolated monophonic stems are evaluated; ensemble mixes are excluded. Each method's search range is tightened to the reference melody's range.

These are real performances, replacing `bach10-mf0-synth` in this suite. The supplied `*-GTF0s.mat` annotations contain fractional MIDI pitches: the loader converts them to Hz, preserves unvoiced frames, and uses the documented 10 ms hop with the first frame centered at 23 ms. These references were produced with YIN and manual corrections, so they are not independent physical measurements of F0. See the [official description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

Data location: `benchmarks/datasets/bach10-original/<piece>/`. The local v1.1 release comes from this [public mirror, pinned revision](https://github.com/flippy-fyp/Bach10_v1.1/tree/2a53cdc6495bb82f03f943b77ca3b11ddf7f5a31). To provision another checkout, extract that archive into the data location, retaining the ten piece directories. Missing original data raises an error; synthesized audio is never substituted.

This cell runs all configured available methods, then resumes cached results on subsequent runs. Original results use `preliminary_bach10_original`; historical synthesized results under `preliminary` remain unchanged.


In [6]:
preliminary = BENCH.run_preliminary(cache_methods=CACHE_METHODS)


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9697,0.9701,0.9280,0.9787,0.3948
attune,0.9543,0.9548,0.9276,0.9614,0.2796
basic_pitch,0.9142,0.9154,0.8947,0.9552,0.2567
praat,0.9587,0.9594,0.9358,0.9744,0.2414
swiftf0,0.9351,0.9351,0.9051,0.9467,0.3272
crepe,0.9699,0.9704,0.9348,0.9787,0.3374
spice,0.8849,0.8866,0.8543,0.9199,0.3829
rmvpe,0.9437,0.9470,0.9142,0.9615,0.3136
mcleod,0.9263,0.9292,0.8950,0.9499,0.3481


## Preliminary URMP comparison

Run the same detector set on two seed-selected isolated stems from each of URMP's 13 instrument classes (26 total). The adapter pairs each `AuSep` WAV with its 10 ms `F0s` annotation and carries the piece, ensemble size, instrument, and part number into the result rows. This is the quickest balanced check that the downloaded corpus and every installed method work end to end.

In [7]:
preliminary_urmp = BENCH.run_preliminary_urmp(force=False, cache_methods=CACHE_METHODS)


Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  21/21] attune: 43_Chorale_tpt_tpt_hn_tbn_tba/AuSep_4_tbn_43_Chorale 


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9545,0.9623,0.8569,0.9843,0.4688
attune,0.9392,0.9469,0.9291,0.9648,0.1048
basic_pitch,0.9021,0.9045,0.8733,0.9776,0.2227
praat,0.9237,0.9361,0.9181,0.9532,0.1005
swiftf0,0.8431,0.8461,0.8342,0.8640,0.1955
crepe,0.9298,0.9341,0.9189,0.9476,0.1173
spice,0.8861,0.8879,0.8518,0.9446,0.2625
rmvpe,0.9228,0.9353,0.8846,0.9625,0.2427
mcleod,0.9364,0.9514,0.8930,0.9728,0.2519


### Low-latency real-time replay

This live comparison has one controlled pair, one explicitly frame-restarted local pYIN diagnostic, and two neural adapters. Attune's ordinary-candidate frame-local pYIN with the selected causal `0.9875` confidence and `0.06 × running-p95 RMS` gates and raw Praat receive the identical 44.1-kHz capture frame, hop, center timestamp, and score-derived nominal pitch range. `pyin_framewise` runs the local implementation on each newly available 4096-sample, left-aligned frame (`center=False`) at 44.1 kHz. It uses the reference single-frame joint voiced/unvoiced state decision, without an extra 0.5 probability gate or temporal smoothing; it remains distinct from the full-track `pyin` baseline.

SwiftF0 and SPICE use their smallest practical model-native rolling center frames; every update commits one estimate only after its required future samples arrive. SwiftF0 uses 320 ms of receptive context / about 152 ms look-ahead at 16 ms updates, SPICE 64 ms / 32 ms at 32 ms updates. RMVPE is excluded from online runs because its pilot configuration accumulated backlog; it remains in the offline comparison. There is no whole-file call, 1--2 second bounded-history rerun, or timestamp ablation in this table. Model loading and one warm-up call happen before steady-state timing.

The first table uses the notebook's standard pitch-summary columns. The second retains per-recording interaction diagnostics with the same title-case naming convention, and the third shows each adapter's native geometry while making the Attune/Praat match explicit and the frame-restarted librosa geometry visible. Attune uses a `0.06` running-p95 RMS floor in live mode and a `0.06` whole-track p95 RMS floor post hoc. Pass `instruments=()` to sample one stem from every selected corpus instrument, and set `force=True` only when intentionally replacing otherwise current rows.

**Latency retention fix (2026-10-04):** v7 saved only per-recording percentiles, so its exact pooled P95 cannot be recovered. v9 saves every output-update latency, CPU duration, and estimate timestamp in compressed checkpoint sidecars. Ordinary reruns now use a new versioned folder and rerun all methods once; completed v9 recordings are reused only while their latency samples remain valid. Existing v7 results are preserved. Restart the kernel before rerunning after this code update.

Methods use up to all but two logical CPUs (`CONFIG.streaming_workers` sets the ceiling), with concurrency reduced according to available RAM before each method, reusing warmed models within each worker. Progress shows the latest completed track with an elapsed timer. Each completed song/method is checkpointed immediately; rerun with `force=False` to resume, or `force=True` to recompute. Frames remain sequential. Like `pyin_ablation.ipynb`, output latency and deadline misses use dedicated-worker process CPU simulation; measured wall compute time remains a separate diagnostic and can increase under contention. The timing cache version separates these results from the older wall-latency rows.

**pYIN correction (2026-10-04):** v9 removes the nonstandard 50% voiced-probability cutoff. The single-frame decision compares the strongest voiced state with one unvoiced state, matching reference one-frame Viterbi with uniform initialization. Restart the kernel and rerun the online cells; `force=False` uses fresh v9 output folders and preserves older results. Previous streaming pYIN accuracy and paired comparisons must be replaced.


The scheduler leaves two logical CPUs free and reserves the larger of 2 GiB or 15% of physical RAM for the desktop. Each method starts with one calibration job, then budgets measured peak worker memory plus 30% (at least 0.5 GiB per worker). Before a real detection has completed, unmeasured heavy methods retain a 2 GiB budget; cached predictions do not lower it. Offline jobs contain at most four tracks (one for CREPE). CREPE divides the CPU budget across its TensorFlow inference pools: with an eight-core budget, one RAM-limited worker can use eight threads, or two workers can use four threads each. Other methods retain one numerical thread per worker. New CREPE caches record `inference_threads`; older caches remain reusable and retain their original timing. CPU timing still sums process CPU time across threads, so it is not wall throughput. Streaming pools reuse warmed models for batches of eight tracks per worker. Pools release their memory and recheck available RAM between batches. If available RAM falls below the desktop reserve at a batch boundary, execution stops with a resumable error. These are scheduling estimates, not hard memory limits.

**McLeod / Tartini 1.2:** `mcleod` now executes the official C++ MPM engine with its default 2048-sample window and 1024-sample hop at 44.1 kHz. It retains persistent filtering, noise-floor scoring, note tracking, warped NSDF aggregation and retrospective octave correction. Streaming commits causal estimates with 23.22 ms look-ahead; offline reads revised history on the native grid. The common score-range gate applies after inference. See `benchmarks/modules/pitch/competitors/tartini/README.md` for the pinned source, build requirements, explicit fixes, and why the later thesis pipeline differs from the linked release. Timing excludes GUI/device I/O.


In [8]:
streaming_urmp = BENCH.run_streaming_comparison(
    instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)


Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/21] attune: 0/21 finished | 0s | 0 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  21/21] attune: 20/21 finished | 217s | finished: 34_Fugue_tpt_tpt_hn_tbn/AuSep_3_hn_34_Fugue         


## Preliminary CocoChorales comparison

The same methods over two seed-selected stems from each of CocoChorales' 13 instrument classes (26 total). The helper prefers two different source compositions within each class and materializes the selected stems automatically. Rows carry ensemble/instrument metadata, and each search range is taken from a robust percentile of the annotated f0 rather than its extremes.

In [9]:
preliminary_coco = BENCH.run_preliminary_coco(force=False, cache_methods=CACHE_METHODS)


materialized 0 file(s) for test
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  26/26] attune: test__woodwind_track228859__2_oboe     


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9648,0.9673,0.9500,0.9868,0.3191
attune,0.9612,0.9636,0.9557,0.9798,0.1434
basic_pitch,0.9658,0.9663,0.9462,0.9959,0.4086
praat,0.9474,0.9575,0.9423,0.9836,0.1491
swiftf0,0.9057,0.9062,0.8954,0.9276,0.2911
crepe,0.9617,0.9633,0.9507,0.9832,0.2489
spice,0.9352,0.9363,0.9215,0.9722,0.3283
rmvpe,0.9260,0.9270,0.9202,0.9680,0.1848
mcleod,0.8479,0.8515,0.8516,0.8710,0.0797


## Offline paired analysis

Pool all five metrics across the eight Bach10, 26 URMP, and 26 CocoChorales preliminary tracks. Compare Attune with each offline competitor on pooled overall accuracy using paired source-piece label swaps and 95% paired cluster-bootstrap intervals. Apply Holm correction within the offline family. Every comparison and the headline table use the same completed track intersection; coverage and excluded tracks are reported. Missing methods raise an error rather than silently changing the test family.

Frames retain equal scoring weight, while dependent frames and stems stay together for inference. `source_groups` can link known shared compositions across corpora. Intervals are unadjusted; adjusted p-values determine significance. These development samples do not establish held-out generalization.


In [10]:
offline_paired = BENCH.show_paired_analysis(
    preliminary,
    preliminary_urmp,
    preliminary_coco,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


Offline: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,6,6,20582,18229,2353
coco,26,26,53986,51162,2824
urmp,21,12,182832,140681,42151


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9584,0.9642,0.8821,0.9844,0.4562
attune,0.9459,0.9516,0.9345,0.9682,0.1158
basic_pitch,0.9187,0.9205,0.8903,0.9801,0.2355
praat,0.9325,0.9433,0.9246,0.9624,0.1104
swiftf0,0.8663,0.8685,0.8527,0.8867,0.2077
crepe,0.9411,0.9444,0.9269,0.9590,0.1361
spice,0.8979,0.8996,0.8666,0.9492,0.2724
rmvpe,0.9254,0.9343,0.8945,0.9638,0.2428
mcleod,0.9140,0.9252,0.8845,0.9460,0.2464


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin,Overall Accuracy,53,44,0,257400,0,9999,5.2393,2.6304,8.5119,0.0001,9999,0.0008,True
1,basic_pitch,Overall Accuracy,53,44,0,257400,0,9999,4.4200,2.6031,6.3661,0.0001,9999,0.0008,True
2,praat,Overall Accuracy,53,44,0,257400,0,9999,0.9899,0.1008,2.0790,0.0484,9999,0.0968,False
3,swiftf0,Overall Accuracy,53,44,0,257400,0,9999,8.1830,5.1500,13.4184,0.0001,9999,0.0008,True
4,crepe,Overall Accuracy,53,44,0,257400,0,9999,0.7657,0.1262,1.4032,0.0528,9999,0.0968,False
5,spice,Overall Accuracy,53,44,0,257400,0,9999,6.7906,4.2715,10.3166,0.0001,9999,0.0008,True
6,rmvpe,Overall Accuracy,53,44,0,257400,0,9999,4.0054,2.4216,5.6408,0.0002,9999,0.0008,True
7,mcleod,Overall Accuracy,53,44,0,257400,0,9999,5.0035,3.4753,6.6366,0.0001,9999,0.0008,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/offline/c59817ba9831


## Online paired analysis

Add the same eight Bach10 and 26 CocoChorales tracks to the 26-track URMP streaming replay. Pool all five metrics across datasets, then compare overall accuracy, voicing recall, and voicing false-alarm rate against each of the five online competitors. Holm correction covers these fifteen online tests separately from offline tests. Source-piece pairing, confidence intervals, coverage checks, and frame weights follow the offline analysis. The streaming pYIN adaptation is included in all three paired comparisons.


In [11]:
streaming_bach10 = BENCH.run_streaming_comparison(
    dataset="bach10-original", instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)
streaming_coco = BENCH.run_streaming_comparison(
    dataset="coco", instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)
online_paired = BENCH.show_paired_analysis(
    streaming_bach10,
    streaming_urmp,
    streaming_coco,
    mode="Streaming",
    methods=ONLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


Streaming replay: 6 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/6] attune: 0/6 finished | 0s | 0 cached; loading model  attune: 6 worker(s); forced 8-worker policy (limited only by remaining tracks)
[   6/6] attune: 5/6 finished | 33s | finished: 10-NunBitten-saxphone   
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/26] attune: 0/26 finished | 0s | 0 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  26/26] attune: 25/26 finished | 64s | finished: test__woodwind_track229162__4_bassoon  


,recordings,output_updates,pooled_p95_output_latency_ms
model,,,
attune,53,885180,47.1189
pyin_framewise,53,885180,47.8919
praat,53,885180,48.2979
swiftf0,53,159847,163.9462
spice,53,80363,36.9639
mcleod,53,110834,23.3820
crepe_tiny,53,27688,63.8765


Streaming: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,6,6,20582,18229,2353
coco,26,26,53986,51162,2824
urmp,21,12,182831,140681,42150


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
attune,0.9426,0.9501,0.9304,0.9687,0.1237
pyin_framewise,0.9612,0.9695,0.8122,0.9991,0.8494
praat,0.9303,0.9521,0.8517,0.9978,0.4968
swiftf0,0.8629,0.8651,0.8519,0.8829,0.1969
spice,0.8463,0.8595,0.8211,0.9108,0.2907
mcleod,0.9106,0.9223,0.8826,0.9451,0.2419
crepe_tiny,0.8590,0.8659,0.8457,0.9194,0.2132


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin_framewise,Overall Accuracy,53,44,0,257399,0,9999,11.8198,7.8995,15.8782,0.0001,9999,0.0018,True
1,pyin_framewise,Voicing Recall,53,44,0,210072,0,9999,-3.0413,-4.3573,-1.9245,0.0001,9999,0.0018,True
2,pyin_framewise,Voicing False Alarm,53,44,0,47327,0,9999,-72.5696,-77.4413,-65.6380,0.0001,9999,0.0018,True
3,praat,Overall Accuracy,53,44,0,257399,0,9999,7.8641,6.1043,9.7217,0.0001,9999,0.0018,True
4,praat,Voicing Recall,53,44,0,210072,0,9999,-2.9128,-4.1418,-1.8520,0.0001,9999,0.0018,True
5,praat,Voicing False Alarm,53,44,0,47327,0,9999,-37.3064,-50.4098,-25.4185,0.0001,9999,0.0018,True
6,swiftf0,Overall Accuracy,53,44,0,257399,0,9999,7.8466,4.8104,13.2127,0.0001,9999,0.0018,True
7,swiftf0,Voicing Recall,53,44,0,210072,0,9999,8.5785,3.8965,15.6262,0.0001,9999,0.0018,True
8,swiftf0,Voicing False Alarm,53,44,0,47327,0,9999,-7.3151,-15.8946,-0.0915,0.0878,9999,0.0878,False
9,spice,Overall Accuracy,53,44,0,257399,0,9999,10.9278,6.8516,15.8116,0.0001,9999,0.0018,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/streaming/d73aa265e4d4


## Expanded filtered paper-cohort evaluation

The offline cell retains the existing evaluation. To run only the updated online experiment, skip the offline cell and use **Expanded online native-geometry experiment** below: first its dry-run cell, then its execution cell. Both require the setup cells only.

The selection remains 40 Bach10, 149 URMP and 260 CocoChorales (20 per instrument), with existing reference alignment, pitch ranges and scoring tolerances. The new comparison uses the same development benchmark set; it is not new held-out validation.


In [12]:
paper_offline = BENCH.run_paper_evaluation("Offline")
paper_offline_paired = BENCH.show_paired_analysis(
    *paper_offline,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES
)

Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  32/32] attune: 10-NunBitten-violin                    


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9730,0.9738,0.9268,0.9842,0.4406
attune,0.9662,0.9670,0.9350,0.9751,0.3131
basic_pitch,0.9272,0.9279,0.8984,0.9720,0.3308
praat,0.9622,0.9631,0.9375,0.9799,0.2585
swiftf0,0.9497,0.9499,0.9103,0.9632,0.4028
crepe,0.9742,0.9749,0.9340,0.9838,0.3858
spice,0.8811,0.8821,0.8437,0.9279,0.4535
rmvpe,0.9626,0.9638,0.9274,0.9818,0.3524
mcleod,0.9414,0.9427,0.9024,0.9635,0.4081


Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  64/109] attune: 29_Fugue_fl_fl_ob_cl/AuSep_3_ob_29_Fugue              Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 109/109] attune: 44_K515_vn_vn_va_va_vc/AuSep_5_vc_44_K515             


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9574,0.9677,0.8708,0.9857,0.4172
attune,0.9442,0.9543,0.9306,0.9695,0.1146
basic_pitch,0.9235,0.9280,0.8953,0.9835,0.1985
praat,0.9269,0.9405,0.9276,0.9533,0.0701
swiftf0,0.8337,0.8385,0.8320,0.8569,0.1734
crepe,0.9414,0.9484,0.9297,0.9611,0.1093
spice,0.9004,0.9056,0.8739,0.9566,0.2142
rmvpe,0.9209,0.9439,0.8910,0.9669,0.2086
mcleod,0.9474,0.9646,0.9081,0.9810,0.2228


materialized 0 file(s) for test
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  64/260] attune: test__brass_track223823__4_tuba    Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 128/260] attune: test__random_track235324__3_horn       Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 192/260] attune: test__string_track216994__4_cello      Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 256/260] attune: test__woodwind_track229821__3_clarinet Pitch batch (attune): 4/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 260/260] attune: test__woodwind_track229950__4_bassoon  


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9677,0.9705,0.9520,0.9884,0.3288
attune,0.9613,0.9640,0.9550,0.9796,0.1571
basic_pitch,0.9676,0.9685,0.9461,0.9972,0.4390
praat,0.9489,0.9571,0.9429,0.9809,0.1637
swiftf0,0.9156,0.9163,0.9036,0.9379,0.3098
crepe,0.9652,0.9663,0.9524,0.9847,0.2769
spice,0.9367,0.9383,0.9227,0.9757,0.3269
rmvpe,0.9304,0.9319,0.9242,0.9701,0.1873
mcleod,0.8604,0.8639,0.8624,0.8803,0.1019


Offline: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,32,8,104196,92556,11640
coco,260,256,556720,527274,29446
urmp,109,21,1242639,955518,287121


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9617,0.9690,0.8976,0.9865,0.4101
attune,0.9512,0.9583,0.9380,0.9732,0.1254
basic_pitch,0.9385,0.9416,0.9104,0.9874,0.2247
praat,0.9363,0.9474,0.9326,0.9641,0.0852
swiftf0,0.8679,0.8711,0.8573,0.8902,0.1938
crepe,0.9513,0.9560,0.9365,0.9703,0.1342
spice,0.9114,0.9152,0.8865,0.9613,0.2328
rmvpe,0.9265,0.9410,0.9027,0.9688,0.2118
mcleod,0.9179,0.9296,0.8944,0.9463,0.2186


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin,Overall Accuracy,401,285,0,1903555,0,9999,4.0389,2.5430,5.3660,0.0001,9999,0.0008,True
1,basic_pitch,Overall Accuracy,401,285,0,1903555,0,9999,2.7648,2.0652,3.5599,0.0001,9999,0.0008,True
2,praat,Overall Accuracy,401,285,0,1903555,0,9999,0.5417,0.0780,1.1030,0.0174,9999,0.0348,True
3,swiftf0,Overall Accuracy,401,285,0,1903555,0,9999,8.0747,6.5352,9.4097,0.0001,9999,0.0008,True
4,crepe,Overall Accuracy,401,285,0,1903555,0,9999,0.1469,-0.1541,0.4659,0.3699,9999,0.3699,False
5,spice,Overall Accuracy,401,285,0,1903555,0,9999,5.1484,4.3904,5.9123,0.0001,9999,0.0008,True
6,rmvpe,Overall Accuracy,401,285,0,1903555,0,9999,3.5334,2.3429,4.4677,0.0001,9999,0.0008,True
7,mcleod,Overall Accuracy,401,285,0,1903555,0,9999,4.3601,3.2596,5.5956,0.0001,9999,0.0008,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/offline/5b3e3cc0a772


### Expanded online native-geometry experiment (November 2026)

Run the setup cells, then the dry-run and execution cells below to run only the same **449 tracks (40 Bach10, 149 URMP, 260 CocoChorales)**. This explicit selective policy supersedes `attune_only_refresh` for this experiment. Keep `force_rerun=False`. Historical online and all offline results remain separate.

Only pYIN (4096/1024 samples at 44.1 kHz; joint frame decision without sequence HMM), Praat (native one-frame raw AC, 3/floor window and 0.75/floor hop, no cross-frame tracking), and CREPE-tiny (64 ms windows / 10 ms updates, no Viterbi) get new inference identities. Praat uses the shared range directly, without the app wrapper's padding or 40 Hz clamp; its silence reference is buffer-local. Attune running-p95, SwiftF0, SPICE and native Tartini must reuse compatible checkpoints. Their missing or incompatible evidence blocks the run **before any inference**. Neural models retain their fixed inference ranges and common final output range gate.

The dry-run saves exact per-track misses and reasons. Execution repeats preflight, resumes completed method/track checkpoints, and returns the new accuracy and **P95 feedback latency (ms)** table. This is the duration-weighted steady-state age of the latest available estimate, including held outputs, analysis look-ahead and simulated process-CPU computation/queueing. Startup and device/rendering delays are excluded; playback ends at audio duration without a drain period. Timing coverage and startup diagnostics, per-track geometry, derived metric caches and provenance are saved alongside the report. Paired tests below are recomputed from the returned rows. This is a configuration comparison on the already-used set, not held-out validation.


In [13]:
# Coverage preview; execution refreshes only Attune.
native_feedback_plan = BENCH.run_native_feedback_evaluation(dry_run=True)
display(native_feedback_plan.groupby(['dataset', 'model']).size().rename('retained_tracks'))


dataset          model         
bach10-original  attune             32
                 crepe_tiny         32
                 mcleod             32
                 praat              32
                 pyin_framewise     32
                 spice              32
                 swiftf0            32
coco             attune            260
                 crepe_tiny        260
                 mcleod            260
                 praat             260
                 pyin_framewise    260
                 spice             260
                 swiftf0           260
urmp             attune            109
                 crepe_tiny        109
                 mcleod            109
                 praat             109
                 pyin_framewise    109
                 spice             109
                 swiftf0           109
Name: retained_tracks, dtype: int64

In [14]:
# Refresh Attune online checkpoints and combine with frozen competitor results.
paper_online, paper_online_table = BENCH.run_native_feedback_evaluation(dry_run=False)
paper_online_paired = BENCH.show_paired_analysis(
    *paper_online,
    mode="Streaming",
    methods=PAPER_ONLINE_METHODS,
)
paper_online_table


Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/32] attune: 0/32 finished | 0s | 0 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  32/32] attune: 31/32 finished | 117s | finished: 10-NunBitten-saxphone                
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/109] attune: 0/109 finished | 0s | 0 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  65/109] attune: 64/109 finished | 770s | finished: 29_Fugue_fl_fl_ob_cl/AuSep_3_ob_29_Fugue                attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[ 109/109] attune: 108/109 finished | 1467s | finished: 44_K515_vn_vn_va_va_vc/AuSep_4_va_44_K515           
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   1/260] attune: 0/260 finished | 

,recordings,output_updates,pooled_p95_output_latency_ms
model,,,
attune,401,6545821,47.1359
crepe_tiny,401,1901263,34.7878
mcleod,401,819603,23.3350
praat,401,5597224,14.5925
pyin_framewise,401,818400,47.1429
spice,401,594298,35.9990
swiftf0,401,1181951,160.0892


Streaming: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,32,8,104196,92556,11640
coco,260,256,556720,527274,29446
urmp,109,21,1242636,955518,287118


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
attune,0.9482,0.9573,0.9340,0.9743,0.1344
pyin_framewise,0.9595,0.9683,0.8315,0.9994,0.7832
praat,0.9527,0.9630,0.9147,0.9938,0.2681
swiftf0,0.8646,0.8677,0.8562,0.8863,0.1843
spice,0.8832,0.8955,0.8549,0.9395,0.2812
mcleod,0.9147,0.9269,0.8924,0.9454,0.2148
crepe_tiny,0.9242,0.9381,0.9144,0.9553,0.1331


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin_framewise,Overall Accuracy,401,285,0,1903552,0,9999,10.2516,7.4973,12.5249,0.0001,9999,0.0018,True
1,pyin_framewise,Voicing Recall,401,285,0,1575348,0,9999,-2.5102,-3.4456,-1.8508,0.0001,9999,0.0018,True
2,pyin_framewise,Voicing False Alarm,401,285,0,328204,0,9999,-64.8840,-71.8844,-59.8833,0.0001,9999,0.0018,True
3,praat,Overall Accuracy,401,285,0,1903552,0,9999,1.9315,1.4181,2.5174,0.0001,9999,0.0018,True
4,praat,Voicing Recall,401,285,0,1575348,0,9999,-1.9537,-2.8008,-1.3722,0.0001,9999,0.0018,True
5,praat,Voicing False Alarm,401,285,0,328204,0,9999,-13.3728,-18.6925,-9.3087,0.0001,9999,0.0018,True
6,swiftf0,Overall Accuracy,401,285,0,1903552,0,9999,7.7816,6.2866,9.0651,0.0001,9999,0.0018,True
7,swiftf0,Voicing Recall,401,285,0,1575348,0,9999,8.7962,6.5200,10.6524,0.0001,9999,0.0018,True
8,swiftf0,Voicing False Alarm,401,285,0,328204,0,9999,-4.9990,-11.2464,-0.6049,0.0113,9999,0.0226,True
9,spice,Overall Accuracy,401,285,0,1903552,0,9999,7.9090,6.7397,9.1638,0.0001,9999,0.0018,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/streaming/ce46a35e5966


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,recordings,timing_coverage_failures,evaluated_seconds,evaluated_duration_coverage,P95 feedback latency (ms)
model,,,,,,,,,,
attune,0.9482,0.9573,0.9340,0.9743,0.1344,401,0,18998.3916,0.9980,49.6127
crepe_tiny,0.9242,0.9381,0.9144,0.9553,0.1331,401,0,19008.9332,0.9986,43.9906
mcleod,0.9147,0.9269,0.8924,0.9454,0.2148,401,0,19026.4856,0.9995,45.3729
praat,0.9527,0.9630,0.9147,0.9938,0.2681,401,0,19029.1633,0.9996,27.1044
pyin_framewise,0.9595,0.9683,0.8315,0.9994,0.7832,401,0,18998.3476,0.9980,69.0413
spice,0.8832,0.8955,0.8549,0.9395,0.2812,401,0,19008.5739,0.9986,66.0997
swiftf0,0.8646,0.8677,0.8562,0.8863,0.1843,401,0,18904.2633,0.9931,175.1790


## Interpreting raw pitch accuracy

Raw pitch accuracy uses a 50-cent tolerance; raw chroma accuracy additionally ignores octave errors. Under these adapters' zero-frequency unvoiced convention, raw pitch accuracy decreases both when played tones are suppressed and when voiced estimates have the wrong pitch. The following decomposition uses pooled reference-voiced frame counts. It is not a mean of track-level percentages. Negative-frequency latent predictions, if introduced by another adapter, would require revisiting this decomposition.


In [15]:
rpa_decomposition = BENCH.show_rpa_decomposition(
    preliminary, preliminary_urmp, preliminary_coco, methods=PRIMARY_METHODS
)


Tracks  Raw Pitch Accuracy  \
Suite                             Method                               
pitch/preliminary_bach10_original pyin         6              0.9697   
                                  attune       6              0.9543   
                                  praat        6              0.9587   
                                  spice        6              0.8849   
pitch/preliminary_urmp            pyin        21              0.9545   
                                  attune      21              0.9392   
                                  praat       21              0.9237   
                                  spice       21              0.8861   
pitch/preliminary_coco            pyin        26              0.9648   
                                  attune      26              0.9612   
                                  praat       26              0.9474   
                                  spice       26              0.9352   

                                          Voicing Recall  \
Suite                             Method                   
pitch/preliminary_bach10_original pyin            0.9787   
                                  attune          0.9614   
                                  praat           0.9744   
                                  spice           0.9199   
pitch/preliminary_urmp            pyin            0.9843   
                                  attune          0.9648   
                                  praat           0.9532   
                                  spice           0.9446   
pitch/preliminary_coco            pyin            0.9868   
                                  attune          0.9798   
                                  praat           0.9836   
                                  spice           0.9722   

                                          Missed-voicing share  \
Suite                             Method                         
pitch/preliminary_bach10_original pyin                  0.0213   
                                  attune                0.0386   
                                  praat                 0.0256   
                                  spice                 0.0801   
pitch/preliminary_urmp            pyin                  0.0157   
                                  attune                0.0352   
                                  praat                 0.0468   
                                  spice                 0.0554   
pitch/preliminary_coco            pyin                  0.0132   
                                  attune                0.0202   
                                  praat                 0.0164   
                                  spice                 0.0278   

                                          Wrong-pitch share  \
Suite                             Method                      
pitch/preliminary_bach10_original pyin               0.0091   
                                  attune             0.0071   
                                  praat              0.0157   
                                  spice              0.0350   
pitch/preliminary_urmp            pyin               0.0298   
                                  attune             0.0256   
                                  praat              0.0294   
                                  spice              0.0585   
pitch/preliminary_coco            pyin               0.0219   
                                  attune             0.0186   
                                  praat              0.0363   
                                  spice              0.0369   

                                          RPA / Voicing Recall  
Suite                             Method                        
pitch/preliminary_bach10_original pyin                  0.9908  
                                  attune                0.9926  
                                  praat                 0.9839  
                                  spice                 0.9

## Full comparisons

These cells drop the preliminary per-instrument sampler and run every track in the configured corpora, including all 40 original Bach10 isolated recordings and all 149 isolated real-recording URMP stems. They are long and cache-backed: jobs run in up to `CONFIG.workers` spawned workers with one numerical-library thread per worker, except CREPE, which shares that CPU budget between processes and TensorFlow inference threads, and any method whose dependency is missing is skipped rather than failing the run.

**Skipped by Run All by default.** Set `RUN_FULL_RUNS = True` in the configuration cell to enable these suites.
The full audio suite contains original Bach10 plus the two MDB synthesized corpora. Its new `all_audio_original_bach10` result directory preserves the previous `all_audio` results.


In [16]:
if RUN_FULL_RUNS:
    all_audio = BENCH.run_all_audio()
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


In [17]:
if RUN_FULL_RUNS:
    all_urmp = BENCH.run_all_urmp(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


In [18]:
if RUN_FULL_RUNS:
    all_coco = BENCH.run_all_coco(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


## Leaderboard

One row per (method, suite), ranked by overall accuracy. Read `Voicing False Alarm` alongside it: the synthetic suites have effectively digital silence in their unvoiced frames, while original Bach10 and URMP supply real isolated recordings. Neither substitutes for a live microphone benchmark in a real room.

With `RUN_FULL_RUNS = False`, this table uses the three preliminary suites.

In [19]:
leaderboard = BENCH.show_leaderboard(
    *(
        (all_audio, all_urmp, all_coco)
        if RUN_FULL_RUNS
        else (preliminary, preliminary_urmp, preliminary_coco)
    )
)


Tracks  Raw Pitch Accuracy  \
model       suite                                                           
attune      pitch/preliminary_coco                 26              0.9612   
crepe       pitch/preliminary_coco                 26              0.9617   
pyin        pitch/preliminary_coco                 26              0.9648   
basic_pitch pitch/preliminary_coco                 26              0.9658   
praat       pitch/preliminary_coco                 26              0.9474   
            pitch/preliminary_bach10_original       6              0.9587   
crepe       pitch/preliminary_bach10_original       6              0.9699   
attune      pitch/preliminary_urmp                 21              0.9392   
pyin        pitch/preliminary_bach10_original       6              0.9697   
attune      pitch/preliminary_bach10_original       6              0.9543   
spice       pitch/preliminary_coco                 26              0.9352   
rmvpe       pitch/preliminary_coco                 26              0.9260   
crepe       pitch/preliminary_urmp                 21              0.9298   
praat       pitch/preliminary_urmp                 21              0.9237   
rmvpe       pitch/preliminary_bach10_original       6              0.9437   
swiftf0     pitch/preliminary_bach10_original       6              0.9351   
            pitch/preliminary_coco                 26              0.9057   
mcleod      pitch/preliminary_bach10_original       6              0.9263   
basic_pitch pitch/preliminary_bach10_original       6              0.9142   
mcleod      pitch/preliminary_urmp                 21              0.9364   
rmvpe       pitch/preliminary_urmp                 21              0.9228   
basic_pitch pitch/preliminary_urmp                 21              0.9021   
pyin        pitch/preliminary_urmp                 21              0.9545   
spice       pitch/preliminary_bach10_original       6              0.8849   
            pitch/preliminary_urmp                 21              0.8861   
mcleod      pitch/preliminary_coco                 26              0.8479   
swiftf0     pitch/preliminary_urmp                 21              0.8431   

                                               Raw Chroma Accuracy  \
model       suite                                                    
attune      pitch/preliminary_coco                          0.9636   
crepe       pitch/preliminary_coco                          0.9633   
pyin        pitch/preliminary_coco                          0.9673   
basic_pitch pitch/preliminary_coco                          0.9663   
praat       pitch/preliminary_coco                          0.9575   
            pitch/preliminary_bach10_original               0.9594   
crepe       pitch/preliminary_bach10_original               0.9704   
attune      pitch/preliminary_urmp                          0.9469   
pyin        pitch/preliminary_bach10_original               0.9701   
attune      pitch/preliminary_bach10_original               0.9548   
spice       pitch/preliminary_coco                          0.9363   
rmvpe       pitch/preliminary_coco                          0.9270   
crepe       pitch/preliminary_urmp                          0.9341   
praat       pitch/preliminary_urmp                          0.9361   
rmvpe       pitch/preliminary_bach10_original               0.9470   
swiftf0     pitch/preliminary_bach10_original               0.9351   
            pitch/preliminary_coco                          0.9062   
mcleod      pitch/preliminary_bach10_original               0.9292   
basic_pitch pitch/preliminary_bach10_original               0.9154   
mcleod      pitch/preliminary_urmp                          0.9514   
rmvpe       pitch/preliminary_urmp                          0.9353   
basic_pitch pitch/preliminary_urmp                          0.9045   
pyin        pitch/preliminary_urmp                          0.9623   
spice       pitch/preliminary_bach10_original               0.8

## Terminal equivalents

The same suites from a shell, for long overnight runs. `--write` publishes clean-suite frame-pooled metrics to `results/pitch/pitch_frame_weighted_benchmarks.csv`. Per-track results are written under `results/pitch/raw_outputs/<method>/`.

```shell
# balanced preliminary Bach10 draw, plus dependency availability
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets bach10-original --per-instrument 2 --seed 0 --list

# full original Bach10 plus MDB synthesized corpora
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets mdb-stem-synth mdb-melody-synth bach10-original --write

# balanced preliminary URMP draw, then the complete 149-stem corpus
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets urmp --per-instrument 2 --seed 0
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets urmp --write

# balanced preliminary CocoChorales draw
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets coco --split test --per-instrument 2 --seed 0 --materialize --write

# the heavy PyTorch models in their own invocation, so runtimes do not stack
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets bach10-original --methods torchcrepe penn --write

# resume an interrupted run, or re-score existing caches without detecting
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --skip-cached --write
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --cache-only --write
```